In [76]:
#%pip install pandas pyarrow
import pandas as pd
import pyarrow.parquet as pq

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


In [58]:
# Load in all stablecoins
dai = pd.read_parquet("../dse4101-project/clean_data/dai_clean.parquet")
pax = pd.read_parquet("../dse4101-project/clean_data/pax_clean.parquet")
usdc = pd.read_parquet("../dse4101-project/clean_data/usdc_clean.parquet")
usdt = pd.read_parquet("../dse4101-project/clean_data/usdt_clean.parquet")
ustd = pd.read_parquet("../dse4101-project/clean_data/ustd_clean.parquet")

In [59]:
print("DAI:", dai["depeg"].sum())
print("USDC:", usdc["depeg"].sum())
print("USDT:", usdt["depeg"].sum())
print("USDP:", pax["depeg"].sum())

ustd_cut = ustd[ustd["timestamp"] <= "2022-05-09"]
print("USTD:", ustd_cut["depeg"].sum())

DAI: 5
USDC: 3
USDT: 2
USDP: 23
USTD: 24


### Identifying optimal train/test split
We define a function below to help identify the optimal split for each stablecoin such that the number of depegs in each split is approximately equal

In [60]:

def split_equal_depegs(df, date_col="timestamp", target_col="depeg"):
    
    df = df.sort_values(date_col).reset_index(drop=True)
    df = df[df["timestamp"] <= "2024-12-31"]

    # cumulative number of depegs
    cumsum = df[target_col].cumsum()
    total = cumsum.iloc[-1]

    # find split where half depegs in each
    split_idx = (cumsum - total/2).abs().idxmin()

    train = df.iloc[:split_idx+1].copy()
    test  = df.iloc[split_idx+1:].copy()

    print("Split date:", df.loc[split_idx, date_col])
    print("Train depegs:", train[target_col].sum())
    print("Test  depegs:", test[target_col].sum())
    print("Train size:", len(train), "Test size:", len(test))
    print("------")

    return train, test

In [61]:
splits = {}

# normal stablecoins
for name, df in {
    "DAI": dai,
    "USDC": usdc,
    "USDT": usdt,
    "USDP": pax
}.items():
    
    print(name)
    train, test = split_equal_depegs(df)
    splits[name] = (train, test)


# TerraUSD (only using period up till May 2022)
print("USTD (TerraUSD)")
ustd_cut = ustd[ustd["timestamp"] <= "2022-05-09"]
ustd_train, ustd_test = split_equal_depegs(ustd_cut)

splits["USTD (TerraUSD)"] = (ustd_train, ustd_test)

DAI
Split date: 2021-11-10 23:59:59
Train depegs: 2
Test  depegs: 3
Train size: 351 Test size: 1146
------
USDC
Split date: 2021-04-17 23:59:59
Train depegs: 1
Test  depegs: 2
Train size: 144 Test size: 1353
------
USDT
Split date: 2021-04-17 23:59:59
Train depegs: 1
Test  depegs: 1
Train size: 144 Test size: 1353
------
USDP
Split date: 2023-07-25 23:59:59
Train depegs: 11
Test  depegs: 12
Train size: 973 Test size: 524
------
USTD (TerraUSD)
Split date: 2021-02-04 23:59:59
Train depegs: 12
Test  depegs: 12
Train size: 72 Test size: 458
------


### Summary table

In [62]:
summary = []

for k,(tr,te) in splits.items():
    summary.append({
        "coin": k,
        "train_start": tr.timestamp.min(),
        "train_end": tr.timestamp.max(),
        "test_start": te.timestamp.min(),
        "test_end": te.timestamp.max(),
        "train_depegs": tr.depeg.sum(),
        "test_depegs": te.depeg.sum()
    })

pd.DataFrame(summary)

,coin,train_start,train_end,test_start,test_end,train_depegs,test_depegs
0,DAI,2020-11-25 23:59:59,2021-11-10 23:59:59,2021-11-11 23:59:59,2024-12-30 23:59:59,2,3
1,USDC,2020-11-25 23:59:59,2021-04-17 23:59:59,2021-04-18 23:59:59,2024-12-30 23:59:59,1,2
2,USDT,2020-11-25 23:59:59,2021-04-17 23:59:59,2021-04-18 23:59:59,2024-12-30 23:59:59,1,1
3,USDP,2020-11-25 23:59:59,2023-07-25 23:59:59,2023-07-26 23:59:59,2024-12-30 23:59:59,11,12
4,USTD (TerraUSD),2020-11-25 23:59:59,2021-02-04 23:59:59,2021-02-05 23:59:59,2022-05-08 23:59:59,12,12


### PCA

In [71]:
df_dai_final = pd.read_parquet("../dse4101-project/clean_data/dai_final.parquet")
df_pax_final = pd.read_parquet("../dse4101-project/clean_data/pax_final.parquet")
df_usdc_final = pd.read_parquet("../dse4101-project/clean_data/usdc_final.parquet")
df_usdt_final = pd.read_parquet("../dse4101-project/clean_data/usdt_final.parquet")
df_ustd_final = pd.read_parquet("../dse4101-project/clean_data/ustd_final.parquet")
# only use ustd data up till May 2022
df_ustd_final = df_ustd_final[df_ustd_final["timestamp"] <= "2022-05-09"]

In [66]:
#Prepare PCA input
def prepare_stablecoin_pca_df(df):

    drop_cols = [
        'depeg', 'timeOpen', 'timeClose', 'timeHigh', 'timeLow',
        'open', 'high', 'low', 'close', 'volume', 'marketCap', 'circulatingSupply',
        'depeg_future_1d', 'depeg_future_3d', 'depeg_future_5d', 'depeg_future_7d', 
        'depeg_future_14d', 'depeg_future_30d'
    ]

    pca_df = df.copy()

    # Drop only columns that actually exist
    pca_df = pca_df.drop(columns=[c for c in drop_cols if c in pca_df.columns], errors="ignore")
    
    pca_df['timestamp'] = pd.to_datetime(pca_df['timestamp']).dt.normalize()
    # Keep only timestamp, symbol and numeric feature columns
    pca_df = pca_df[['timestamp', 'symbol'] + [c for c in pca_df.columns if c not in ['timestamp', 'symbol'] and pd.api.types.is_numeric_dtype(pca_df[c])]]

    # Sort by time
    pca_df = pca_df.sort_values('timestamp').reset_index(drop=True)
    
    return pca_df

In [72]:
# Split each stablecoin into train/test
dai_train, dai_test = split_equal_depegs(df_dai_final)
dai_train_pca = prepare_stablecoin_pca_df(dai_train)
dai_test_pca = prepare_stablecoin_pca_df(dai_test)

pax_train, pax_test = split_equal_depegs(df_pax_final)
pax_train_pca = prepare_stablecoin_pca_df(pax_train)
pax_test_pca = prepare_stablecoin_pca_df(pax_test)

usdc_train, usdc_test = split_equal_depegs(df_usdc_final)
usdc_train_pca = prepare_stablecoin_pca_df(usdc_train)
usdc_test_pca = prepare_stablecoin_pca_df(usdc_test)

usdt_train, usdt_test = split_equal_depegs(df_usdt_final)
usdt_train_pca = prepare_stablecoin_pca_df(usdt_train)
usdt_test_pca = prepare_stablecoin_pca_df(usdt_test)

ustd_train, ustd_test = split_equal_depegs(df_ustd_final)
ustd_train_pca = prepare_stablecoin_pca_df(ustd_train)
ustd_test_pca = prepare_stablecoin_pca_df(ustd_test)


Split date: 2021-11-10 23:59:59
Train depegs: 2
Test  depegs: 3
Train size: 321 Test size: 1146
------
Split date: 2023-07-25 23:59:59
Train depegs: 11
Test  depegs: 12
Train size: 943 Test size: 524
------
Split date: 2021-04-17 23:59:59
Train depegs: 1
Test  depegs: 2
Train size: 114 Test size: 1353
------
Split date: 2021-04-17 23:59:59
Train depegs: 1
Test  depegs: 1
Train size: 114 Test size: 1353
------
Split date: 2021-02-04 23:59:59
Train depegs: 12
Test  depegs: 12
Train size: 42 Test size: 458
------


In [81]:
# run PCA
def fit_fixed_pca(train_df, test_df, n_components):
    numeric_cols = [c for c in train_df.columns if c not in ["timestamp", "symbol"]]

    # Imputer: fill NaNs with training mean
    imputer = SimpleImputer(strategy="mean")
    X_train_imputed = imputer.fit_transform(train_df[numeric_cols])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    
    pca = PCA(n_components=n_components)
    X_train_pcs = pca.fit_transform(X_train_scaled)
    
    # Train results
    train_pcs = pd.DataFrame(X_train_pcs, columns=[f"PC{i+1}" for i in range(X_train_pcs.shape[1])], index=train_df.index)
    
    # Test projection
    X_test_imputed = imputer.transform(test_df[numeric_cols])
    X_test_scaled = scaler.transform(X_test_imputed)
    test_pcs = pd.DataFrame(pca.transform(X_test_scaled), columns=train_pcs.columns, index=test_df.index)
    
    return {
        "loadings": pd.DataFrame(pca.components_.T, index=numeric_cols),
        "explained_variance": pca.explained_variance_ratio_,
        "pc_scores_train": train_pcs,
        "pc_scores_test": test_pcs
    }

In [ ]:
# DAI
dai_pca_result = fit_fixed_pca(dai_train_pca, dai_test_pca, n_components=10)

# USDC
#usdc_pca_result = fit_fixed_pca(usdc_train_pca, usdc_test_pca, n_components=10)

# USTD
#ustd_pca_result = fit_fixed_pca(ustd_train_pca, ustd_test_pca, n_components=10)

# USDT
#usdt_pca_result = fit_fixed_pca(usdt_train_pca, usdt_test_pca, n_components=10)

# USDP (PAX)
#pax_pca_result = fit_fixed_pca(pax_train_pca, pax_test_pca, n_components=10)

In [83]:
dai_pca_result

{'loadings':                                               0         1         2         3  \
 percent_change_24h                     0.021985  0.346382  0.087375  0.083141   
 percent_change_7d                      0.036099  0.353548  0.169733  0.075776   
 percent_change_30d                     0.084231  0.353982  0.227292 -0.059662   
 volume_percent_change_24h             -0.010716 -0.207805  0.208023  0.356664   
 volume_percent_change_7d              -0.058388 -0.151281  0.200954  0.407922   
 volume_percent_change_30d             -0.100873 -0.198476  0.123513  0.358112   
 market_cap_percent_change_24h          0.260527 -0.126569  0.229724  0.243882   
 market_cap_percent_change_7d           0.398002 -0.014121  0.149787 -0.105717   
 market_cap_percent_change_30d          0.377177 -0.053486  0.036085 -0.100472   
 circulating_supply_percent_change_24h  0.263657 -0.094914  0.236946  0.252727   
 circulating_supply_percent_change_7d   0.397260 -0.002652  0.154970 -0.101382   
 cir

In [ ]:
# just save to csv so that i can see the full datasets

#dai_pca_result["pc_scores_train"].to_csv("dai_pca_train.csv", index=False)
#dai_pca_result["pc_scores_test"].to_csv("dai_pca_test.csv", index=False)
#dai_pca_result["loadings"].to_csv("dai_pca_loadings.csv")
#pd.Series(dai_pca_result["explained_variance"]).to_csv("dai_pca_explained_variance.csv")